In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import requests
import json
import glob
from dotenv import load_dotenv

project_root = Path.cwd().parents[1]

# Load API keys from .env (EODHD_API_KEY)
# Get the EODHD API key from Tuleva 1Password and save to .env in the project root
load_dotenv(project_root / '.env')
EODHD_API_KEY = os.environ.get('EODHD_API_KEY')
assert EODHD_API_KEY, 'EODHD_API_KEY not found in .env — get it from Tuleva 1Password'

# ── Parameters ──
SEB_REPORTS_DIR = 'seb_reports'
DEPOSITORY_FEE_RATE = 0.00025  # 0.025% p.a.

FUNDS = [
    {'code': 'TUV100', 'name': 'Tuleva III Samba Pensionifond', 'isin': 'EE3600001707', 'pillar': 3},
    {'code': 'TKF100', 'name': 'Tuleva Täiendav Kogumisfond', 'isin': 'EE0000003283', 'pillar': 3},
]

FUND_FEES = {
    'EE3600001707': 0.00179,  # TUV100: 0.179% p.a.
    'EE0000003283': 0.00152,  # TKF100: 0.152% p.a.
}

# Estonian public holidays (for working day calculations)
ESTONIAN_HOLIDAYS = [
    '2026-01-01', '2026-02-24', '2026-04-03', '2026-04-05',
    '2026-05-01', '2026-05-24', '2026-06-23', '2026-06-24',
    '2026-08-20', '2026-12-24', '2026-12-25', '2026-12-26',
]
EE_HOLIDAYS = np.array(ESTONIAN_HOLIDAYS, dtype='datetime64[D]')

def ee_busdays(date_from, date_to):
    """Count Estonian working days between two dates (exclusive start, inclusive end)."""
    d_from = np.datetime64(date_from, 'D')
    d_to = np.datetime64(date_to, 'D')
    count = int(np.busday_count(d_from, d_to + np.timedelta64(1, 'D'), holidays=EE_HOLIDAYS))
    if np.is_busday(d_from, holidays=EE_HOLIDAYS):
        count -= 1
    return max(count, 1)

def ee_offset(date_str, n=-1):
    """Return the nth Estonian working day relative to date_str."""
    d = np.datetime64(date_str, 'D')
    step = 1 if n > 0 else -1
    remaining = abs(n)
    while remaining > 0:
        d += np.timedelta64(step, 'D')
        if np.is_busday(d, holidays=EE_HOLIDAYS):
            remaining -= 1
    return str(d)

# Auto-discover the two most recent SEB position reports
csv_files = sorted(glob.glob(f'{SEB_REPORTS_DIR}/TULEVA_pos_raport_*.csv'))
assert len(csv_files) >= 1, f'No SEB position reports found in {SEB_REPORTS_DIR}/'

CSV_FILE = csv_files[-1]       # latest
CSV_FILE_PREV = csv_files[-2] if len(csv_files) >= 2 else None

# Extract NAV dates from CSV headers
def extract_nav_date(filepath):
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('As of:'):
                return line.split(';')[1].strip()
    raise ValueError(f'Could not find NAV date in {filepath}')

NAV_DATE = extract_nav_date(CSV_FILE)
NAV_DATE_PREV = extract_nav_date(CSV_FILE_PREV) if CSV_FILE_PREV else None

print(f'NAV date: {NAV_DATE} (from {Path(CSV_FILE).name})')
if NAV_DATE_PREV:
    print(f'Prev NAV date: {NAV_DATE_PREV} (from {Path(CSV_FILE_PREV).name})')
print(f'T-1: {ee_offset(NAV_DATE, -1)}, T-2: {ee_offset(NAV_DATE, -2)}')
print(f'Report run: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

NAV date: 2026-03-09 (from TULEVA_pos_raport_20260309.csv)
Prev NAV date: 2026-03-06 (from TULEVA_pos_raport_20260306.csv)
T-1: 2026-03-06, T-2: 2026-03-05
Report run: 2026-03-10 16:07


In [2]:
from IPython.display import display, HTML
import getpass

report_html = f"""
<h1 style="margin-bottom: 4px">Tuleva III pillar & TKF funds NAV calculation</h1>
<p style="color: #555; margin: 2px 0">
    <strong>NAV date:</strong> {NAV_DATE} &nbsp;|&nbsp;
    <strong>Report run:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M')} by {getpass.getuser()}
</p>
<p style="color: #333; margin-top: 12px; max-width: 720px; line-height: 1.5">
    This report takes fund position data from the SEB bank position report,
    reprices stale pooled fund holdings using fresh EODHD prices,
    cross-checks security prices against EODHD and Yahoo Finance,
    pulls official NAV data from pensionikeskus.ee,
    calculates the net asset value per unit (incl. management and depository fees),
    and verifies the result by comparing the day-over-day NAV change to a benchmark index.
</p>
<hr style="margin-top: 16px">
"""
display(HTML(report_html))

In [3]:
import io, codecs
import yfinance as yf

# ═══════════════════════════════════════════════════════════════
# 1. Parse SEB position report CSVs → raw_positions
# ═══════════════════════════════════════════════════════════════

def parse_seb_csv(filepath, nav_date):
    """Parse SEB bank position report CSV into a DataFrame."""
    rows = []
    with open(filepath, 'r') as f:
        lines = f.readlines()

    for line in lines[6:]:
        parts = line.strip().split(';')
        if len(parts) < 8 or not parts[0]:
            continue

        fund_code = parts[0]
        account = parts[1]
        isin = parts[2]
        name = parts[3]
        qty_str = parts[4]
        price_str = parts[5]
        value_str = parts[7]

        def parse_num(s):
            if not s or s == '':
                return None
            return float(s.replace('.', '').replace(',', '.'))

        if account == 'Total' or account == '':
            continue

        if account == 'Register':
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'UNITS', 'Account ID': isin,
                'Account Name': name, 'Quantity': parse_num(qty_str),
                'Market Price': None, 'Market Value': None,
            })
        elif account.startswith('VP'):
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'SECURITY', 'Account ID': isin,
                'Account Name': name, 'Quantity': parse_num(qty_str),
                'Market Price': parse_num(price_str), 'Market Value': parse_num(value_str),
            })
        elif 'Cash account' in name:
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'CASH', 'Account ID': '',
                'Account Name': name, 'Quantity': parse_num(value_str),
                'Market Price': 1.0, 'Market Value': parse_num(value_str),
            })
        elif 'receivable' in name.lower():
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'RECEIVABLES', 'Account ID': isin or '',
                'Account Name': name, 'Quantity': None,
                'Market Price': None, 'Market Value': parse_num(value_str),
            })
        elif 'payable' in name.lower():
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'LIABILITY', 'Account ID': isin or '',
                'Account Name': name, 'Quantity': None,
                'Market Price': None, 'Market Value': parse_num(value_str),
            })

    return pd.DataFrame(rows)

# Parse today's and previous day's position reports
raw_positions = parse_seb_csv(CSV_FILE, NAV_DATE)
print(f'Positions (today): {len(raw_positions)} rows from {Path(CSV_FILE).name}')

if CSV_FILE_PREV:
    raw_positions_prev = parse_seb_csv(CSV_FILE_PREV, NAV_DATE_PREV)
    raw_positions = pd.concat([raw_positions, raw_positions_prev], ignore_index=True)
    print(f'Positions (prev):  {len(raw_positions_prev)} rows from {Path(CSV_FILE_PREV).name}')
print(f'Positions total:   {len(raw_positions)} rows')

# ═══════════════════════════════════════════════════════════════
# 2. Fetch ETF/fund prices from EODHD + Yahoo Finance → raw_index
#    Each row: Key, Date (date price applies to), Value, DailyChangePct
# ═══════════════════════════════════════════════════════════════

# EODHD tickers for all holdings across TUV100 and TKF100
EODHD_TICKERS = {
    # Pooled IE funds (stale in SEB — need fresh prices)
    'IE0009FT4LX4.EUFUND': 'IE0009FT4LX4',
    'IE00BFG1TM61.EUFUND': 'IE00BFG1TM61',
    'IE00BKPTWY98.EUFUND': 'IE00BKPTWY98',
    # TUV100 exchange-traded ETFs
    'SGAS.XETRA': 'IE00BFNM3G45',
    'SLMC.XETRA': 'IE00BFNM3D14',
    'SGAJ.XETRA': 'IE00BFNM3L97',
    # TKF100 exchange-traded ETFs
    'XRSM.XETRA': 'IE00BJZ2DC62',
    'V3YA.XETRA': 'IE000O58J820',
    'ESGM.XETRA': 'IE00BMDBMY19',
    'PAC.XETRA': 'LU1291106356',
    'EJAP.XETRA': 'LU1291102447',
    'EEUX.XETRA': 'LU1291099718',
    'D5BH.XETRA': 'LU0476289540',
}

# Pooled IE non-ETF ISINs: SEB price is stale (T-1), need fresh price for NAV calc
STALE_ISINS = {
    'IE0009FT4LX4', 'IE00BFG1TM61', 'IE00BKPTWY98',
}

# Reverse map: ISIN → EODHD key
ISIN_TO_EODHD = {isin: key for key, isin in EODHD_TICKERS.items() if not isin.endswith('_BENCHMARK')}

nav_dt = pd.to_datetime(NAV_DATE)
fetch_start = (nav_dt - pd.Timedelta(days=14)).strftime('%Y-%m-%d')
fetch_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

index_rows = []

# ── EODHD EOD Historical API ──
print('Fetching EODHD historical prices...')
eodhd_ok = 0
for eodhd_key, isin in EODHD_TICKERS.items():
    try:
        code, exchange = eodhd_key.split('.')
        url = f'https://eodhd.com/api/eod/{code}.{exchange}?from={fetch_start}&to={fetch_end}&api_token={EODHD_API_KEY}&fmt=json'
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            for row in data:
                index_rows.append({'Key': eodhd_key, 'Date': row['date'], 'Value': float(row['adjusted_close'])})
            if data:
                eodhd_ok += 1
        else:
            print(f'  EODHD {eodhd_key}: HTTP {resp.status_code}')
    except Exception as e:
        print(f'  EODHD {eodhd_key}: {e}')

print(f'  EODHD: {eodhd_ok}/{len(EODHD_TICKERS)} tickers OK')

# ── Yahoo Finance (backup for ETFs + IG3A benchmark) ──
YAHOO_TICKERS = {
    'SGAS.DE': 'SGAS.XETRA',
    'SLMC.DE': 'SLMC.XETRA',
    'SGAJ.DE': 'SGAJ.XETRA',
    'XRSM.DE': 'XRSM.XETRA',
    'V3YA.DE': 'V3YA.XETRA',
    'ESGM.DE': 'ESGM.XETRA',
    'PAC.DE': 'PAC.XETRA',
    'EJAP.DE': 'EJAP.XETRA',
    'EEUX.DE': 'EEUX.XETRA',
    'D5BH.DE': 'D5BH.XETRA',
    'IG3A.DE': 'IG3A_YAHOO',
}
for yticker, key in YAHOO_TICKERS.items():
    try:
        hist = yf.Ticker(yticker).history(start=fetch_start, end=fetch_end)
        hist.index = hist.index.tz_localize(None)
        for dt, row in hist.iterrows():
            index_rows.append({'Key': key, 'Date': dt.strftime('%Y-%m-%d'), 'Value': float(row['Close'])})
    except Exception as e:
        print(f'  Yahoo {yticker}: {e}')

# ── MSCI indices ──
import ssl
from urllib.request import urlopen
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

msci_start = (nav_dt - pd.Timedelta(days=14)).strftime('%Y%m%d')
msci_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y%m%d')

MSCI_INDICES = {
    'MSCI_ACWI_SCREENED': 722376,
}

for key, idx_code in MSCI_INDICES.items():
    try:
        msci_url = f'https://app2.msci.com/products/service/index/indexmaster/getLevelDataForGraph?currency_symbol=EUR&index_variant=NETR&start_date={msci_start}&end_date={msci_end}&data_frequency=DAILY&index_codes={idx_code}'
        msci_raw = json.loads(urlopen(msci_url, context=ssl_ctx).read())
        n = 0
        for lvl in msci_raw['indexes']['INDEX_LEVELS']:
            dt = pd.to_datetime(str(lvl['calc_date']), format='%Y%m%d')
            index_rows.append({'Key': key, 'Date': dt.strftime('%Y-%m-%d'), 'Value': float(lvl['level_eod'])})
            n += 1
        print(f'  {key}: {n} days')
    except Exception as e:
        print(f'  {key}: {e}')

raw_index = pd.DataFrame(index_rows)
raw_index['Date'] = pd.to_datetime(raw_index['Date'])
raw_index = raw_index.drop_duplicates(subset=['Key', 'Date'], keep='first')
raw_index = raw_index.sort_values(['Key', 'Date'])

# Compute daily % change within each price series
raw_index['DailyChangePct'] = raw_index.groupby('Key')['Value'].pct_change() * 100

print(f'Index data: {len(raw_index)} total rows')

for key in sorted(raw_index['Key'].unique()):
    s = raw_index[raw_index['Key'] == key].sort_values('Date')
    last = s.iloc[-1]
    chg_str = f'{last["DailyChangePct"]:+.2f}%' if pd.notna(last['DailyChangePct']) else 'n/a'
    print(f'  {key}: latest {last["Date"].strftime("%Y-%m-%d")} = {last["Value"]:.4f} (daily chg {chg_str})')

# ═══════════════════════════════════════════════════════════════
# 3. Fetch fund NAVs from pensionikeskus.ee → raw_units
# ═══════════════════════════════════════════════════════════════

def fetch_pensionikeskus(date_str, pillar=2):
    if pillar == 2:
        url = f'https://www.pensionikeskus.ee/statistika/ii-sammas/kogumispensioni-paevastatistika/?date={date_str}&download=xls'
    else:
        url = f'https://www.pensionikeskus.ee/statistika/iii-sammas/taiendava-kogumispensioni-statistika/?date={date_str}&download=xls'

    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    text = resp.content.decode('utf-16-le').lstrip('\ufeff')
    lines = text.strip().split('\n')

    rows = []
    for line in lines[1:]:
        cols = line.strip().split('\t')
        if len(cols) < 4 or not cols[0]:
            continue
        fund_name = cols[0].strip()
        date_col = cols[1].strip()
        nav_str = cols[2].strip()
        maht_str = cols[-1].strip() if len(cols) > 10 else ''

        try:
            nav_val = float(nav_str.replace(',', '.'))
            dt = datetime.strptime(date_col, '%d.%m.%Y')
            actual_date = dt.strftime('%Y-%m-%d')
            maht_val = float(maht_str.replace(',', '.')) * 1_000_000 if maht_str else None
        except (ValueError, IndexError):
            continue

        rows.append({
            'Security Name': fund_name,
            'Request Date': actual_date,
            'Nav': nav_val,
            'Balance': maht_val,
        })
    return rows

units_rows = []
dates_to_fetch = set()
for fund in FUNDS:
    dates_to_fetch.add((NAV_DATE, fund['pillar']))
    for i in range(1, 4):
        dates_to_fetch.add((ee_offset(NAV_DATE, -i), fund['pillar']))

print('Fetching pensionikeskus NAVs...')
fetched_dates = set()
for date_str, pillar in sorted(dates_to_fetch):
    cache_key = (date_str, pillar)
    if cache_key in fetched_dates:
        continue
    try:
        day_rows = fetch_pensionikeskus(date_str, pillar)
        units_rows.extend(day_rows)
        fetched_dates.add(cache_key)
    except Exception as e:
        print(f'  {date_str} (pillar {pillar}): {e}')

raw_units = pd.DataFrame(units_rows)
if len(raw_units) > 0:
    raw_units = raw_units.drop_duplicates(subset=['Security Name', 'Request Date'])
print(f'Units data: {len(raw_units)} rows from pensionikeskus.ee')

# ═══════════════════════════════════════════════════════════════
# 4. Registry data (hardcoded)
# ═══════════════════════════════════════════════════════════════

raw_registry = pd.DataFrame([
    {'Isin': isin, 'Management Fee Rate': rate}
    for isin, rate in FUND_FEES.items()
])
print(f'Registry: {len(raw_registry)} funds')

# ═══════════════════════════════════════════════════════════════
# Prepare per-fund data
# ═══════════════════════════════════════════════════════════════

fund_data = {}
for fund in FUNDS:
    fd = {}
    code = fund['code']

    all_positions = raw_positions[
        (raw_positions['Fund Code'] == code) &
        (raw_positions['Nav Date'] == NAV_DATE)
    ].copy()

    fd['all_positions'] = all_positions
    fd['securities'] = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
    fd['cash'] = all_positions[all_positions['Account Type'] == 'CASH'].copy()
    fd['receivables'] = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
    fd['liabilities'] = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
    fd['units_row'] = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

    print(f'\n{code} on {NAV_DATE}: {len(all_positions)} rows  '
          f'({len(fd["securities"])} sec, {len(fd["cash"])} cash, '
          f'{len(fd["receivables"])} recv, {len(fd["liabilities"])} liab, {len(fd["units_row"])} units)')

    # Units data from pensionikeskus
    all_fund_units = raw_units[raw_units['Security Name'] == fund['name']].copy()
    all_fund_units = all_fund_units.sort_values('Request Date', ascending=False)
    fd['units_today'] = all_fund_units[all_fund_units['Request Date'] == NAV_DATE]

    prev_date = None
    if len(all_fund_units) > 0:
        prev_dates = all_fund_units[all_fund_units['Request Date'] < NAV_DATE]['Request Date'].unique()
        if len(prev_dates) > 0 and len(fd['units_today']) > 0:
            today_nav_val = fd['units_today'].iloc[0]['Nav']
            for d in sorted(prev_dates, reverse=True):
                row = all_fund_units[all_fund_units['Request Date'] == d].iloc[0]
                if row['Nav'] != today_nav_val:
                    prev_date = d
                    break
            if prev_date is None:
                prev_date = sorted(prev_dates)[-1]

    # Fallback: if pensionikeskus has no data for this fund, use SEB prev report date
    if prev_date is None and NAV_DATE_PREV:
        prev_date = NAV_DATE_PREV
        print(f'  Units: not found on pensionikeskus — using SEB prev date {prev_date}')

    fd['prev_date'] = prev_date
    fd['units_prev'] = all_fund_units[all_fund_units['Request Date'] == prev_date] if prev_date and len(all_fund_units) > 0 else pd.DataFrame()
    print(f'  Units dates: today={NAV_DATE}, prev={prev_date}')

    fund_reg = raw_registry[raw_registry['Isin'] == fund['isin']]
    if len(fund_reg) > 0:
        fd['mgmt_fee_rate'] = fund_reg.iloc[0]['Management Fee Rate']
        print(f'  Mgmt fee: {fd["mgmt_fee_rate"]*100:.3f}% p.a.')
    else:
        fd['mgmt_fee_rate'] = None
        print(f'  WARNING: Fund not found in registry!')

    print(f'  Depository fee: {DEPOSITORY_FEE_RATE*100:.3f}% p.a.')

    fund_data[code] = fd

Positions (today): 49 rows from TULEVA_pos_raport_20260309.csv
Positions (prev):  49 rows from TULEVA_pos_raport_20260306.csv
Positions total:   98 rows
Fetching EODHD historical prices...


  EODHD: 13/13 tickers OK


  MSCI_ACWI_SCREENED: 10 days
Index data: 161 total rows
  D5BH.XETRA: latest 2026-03-09 = 103.6000 (daily chg -0.79%)
  EEUX.XETRA: latest 2026-03-09 = 18.6080 (daily chg -0.56%)
  EJAP.XETRA: latest 2026-03-09 = 17.8660 (daily chg -0.43%)
  ESGM.XETRA: latest 2026-03-09 = 41.6150 (daily chg +0.24%)
  IE0009FT4LX4.EUFUND: latest 2026-03-06 = 15.2390 (daily chg -1.19%)
  IE00BFG1TM61.EUFUND: latest 2026-03-06 = 33.8700 (daily chg -1.23%)
  IE00BKPTWY98.EUFUND: latest 2026-03-06 = 13.0600 (daily chg -0.20%)
  IG3A_YAHOO: latest 2026-03-09 = 4.4210 (daily chg -0.45%)
  MSCI_ACWI_SCREENED: latest 2026-03-09 = 4851.9471 (daily chg -1.35%)
  PAC.XETRA: latest 2026-03-09 = 16.2480 (daily chg -0.02%)
  SGAJ.XETRA: latest 2026-03-09 = 7.3880 (daily chg -0.55%)
  SGAS.XETRA: latest 2026-03-09 = 11.9080 (daily chg -0.48%)
  SLMC.XETRA: latest 2026-03-09 = 9.9280 (daily chg -0.76%)
  V3YA.XETRA: latest 2026-03-09 = 6.7920 (daily chg -0.57%)
  XRSM.XETRA: latest 2026-03-09 = 49.7700 (daily chg -0.

Units data: 68 rows from pensionikeskus.ee
Registry: 2 funds

TUV100 on 2026-03-09: 12 rows  (6 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-09, prev=2026-03-06
  Mgmt fee: 0.179% p.a.
  Depository fee: 0.025% p.a.

TKF100 on 2026-03-09: 15 rows  (9 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units: not found on pensionikeskus — using SEB prev date 2026-03-06
  Units dates: today=2026-03-09, prev=2026-03-06
  Mgmt fee: 0.152% p.a.
  Depository fee: 0.025% p.a.


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [4]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']
    cash = fd['cash']
    receivables = fd['receivables']
    liabilities = fd['liabilities']
    units_row = fd['units_row']

    sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
    sec_display = sec_display.sort_values('Market Value', ascending=False)
    sec_total = sec_display['Market Value'].sum()

    cash_total = cash['Market Value'].sum()
    recv_total = receivables['Market Value'].sum()
    liab_total = liabilities['Market Value'].sum()
    other_rows = [
        {'Account ID': '', 'Account Name': 'Cash', 'Quantity': None, 'Market Price': None, 'Market Value': cash_total},
        {'Account ID': '', 'Account Name': 'Receivables', 'Quantity': None, 'Market Price': None, 'Market Value': recv_total},
        {'Account ID': '', 'Account Name': 'Liabilities', 'Quantity': None, 'Market Price': None, 'Market Value': liab_total},
    ]

    all_display = pd.concat([sec_display, pd.DataFrame(other_rows)], ignore_index=True)

    available_dates = sorted(raw_positions[raw_positions['Fund Code'] == code]['Nav Date'].unique())
    nav_idx = available_dates.index(NAV_DATE) if NAV_DATE in available_dates else -1
    prev_nav_date = available_dates[nav_idx - 1] if nav_idx > 0 else None

    if prev_nav_date:
        prev_pos = raw_positions[
            (raw_positions['Fund Code'] == code) &
            (raw_positions['Nav Date'] == prev_nav_date)
        ]
        prev_sec = prev_pos[prev_pos['Account Type'] == 'SECURITY'].set_index('Account ID')['Market Value']
        prev_cash = prev_pos[prev_pos['Account Type'] == 'CASH']['Market Value'].sum()
        prev_recv = prev_pos[prev_pos['Account Type'] == 'RECEIVABLES']['Market Value'].sum()
        prev_liab = prev_pos[prev_pos['Account Type'] == 'LIABILITY']['Market Value'].sum()
        prev_by_type = {'Cash': prev_cash, 'Receivables': prev_recv, 'Liabilities': prev_liab}

        prev_values = []
        for _, row in all_display.iterrows():
            if row['Account ID'] and row['Account ID'] in prev_sec.index:
                prev_values.append(prev_sec[row['Account ID']])
            elif row['Account Name'] in prev_by_type:
                prev_values.append(prev_by_type[row['Account Name']])
            else:
                prev_values.append(None)

        all_display['Prev Value'] = prev_values
        all_display['Change %'] = (all_display['Market Value'] - all_display['Prev Value']) / all_display['Prev Value'].abs() * 100
    else:
        prev_pos = pd.DataFrame()
        all_display['Prev Value'] = None
        all_display['Change %'] = None

    gross_portfolio = all_display['Market Value'].abs().sum()
    all_display['% of Portfolio'] = all_display['Market Value'] / gross_portfolio * 100

    nav_components = sec_total + cash_total + recv_total + liab_total
    prev_nav_total = all_display['Prev Value'].sum() if prev_nav_date else None
    nav_change_pct = (nav_components - prev_nav_total) / abs(prev_nav_total) * 100 if prev_nav_total else None

    total_row = pd.DataFrame([{
        'Account ID': '', 'Account Name': 'Total net assets', 'Quantity': None,
        'Market Price': None, 'Market Value': nav_components,
        'Prev Value': prev_nav_total, 'Change %': nav_change_pct,
        '% of Portfolio': 100.0
    }])
    all_display = pd.concat([all_display, total_row], ignore_index=True)

    position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

    def bold_total(row):
        if row['Account Name'] == 'Total net assets':
            return ['font-weight: bold'] * len(row)
        return [''] * len(row)

    display(all_display[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value', '% of Portfolio', 'Change %']].style
        .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}',
                 '% of Portfolio': '{:.2f}%', 'Change %': '{:+.2f}%'}, na_rep='')
        .set_caption(f'{code} position as of {NAV_DATE} (vs {prev_nav_date})')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(bold_total, axis=1)
        .hide(axis='index'))

    fd['sec_total'] = sec_total
    fd['cash_total'] = cash_total
    fd['recv_total'] = recv_total
    fd['liab_total'] = liab_total
    fd['nav_components'] = nav_components
    fd['prev_nav_date'] = prev_nav_date
    fd['prev_nav_total'] = prev_nav_total
    fd['prev_pos'] = prev_pos if prev_nav_date else pd.DataFrame()
    fd['position_units'] = position_units

Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"9,039,054.83",15.2390,"137,746,156.54",28.94%,-1.19%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"4,053,375.60",33.8700,"137,287,831.57",28.84%,-1.23%
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"9,135,951.00",11.9080,"108,790,904.51",22.86%,-0.48%
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"3,848,906.59",13.0600,"50,266,720.07",10.56%,-0.20%
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"3,585,405.00",9.9280,"35,595,900.84",7.48%,-0.76%
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"520,422.00",7.3880,"3,844,877.74",0.81%,-0.55%
,Cash,,,"2,019,644.31",0.42%,+21.03%
,Receivables,,,"405,429.53",0.09%,-1.35%
,Liabilities,,,0.00,0.00%,
,Total net assets,,,"475,957,465.11",100.00%,-0.82%


Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,"291,716.00",4.3040,"1,255,545.66",18.58%,-0.58%
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,"23,687.00",49.7700,"1,178,901.99",17.45%,-0.56%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"34,096.56",33.8700,"1,154,850.49",17.09%,-1.23%
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,"153,949.00",6.7920,"1,045,621.61",15.47%,-0.57%
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,"45,082.00",18.6080,"838,885.86",12.41%,-0.56%
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,"15,543.00",41.6150,"646,821.95",9.57%,+0.24%
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,"11,704.00",17.8660,"209,103.66",3.09%,-0.43%
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,"6,719.00",16.2480,"109,170.31",1.62%,-0.02%
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,908.00,103.6000,"94,068.80",1.39%,-0.79%
,Cash,,,"224,230.39",3.32%,+6.01%


## Price Verification & Repricing

Cross-checking fund position prices against EODHD and Yahoo Finance. Pooled IE funds (marked with *) have stale prices in SEB — they are repriced using the latest available provider price for the NAV date.

In [5]:
KEY_MAP = {
    # Pooled IE funds (stale in SEB)
    'IE0009FT4LX4': {'EODHD': 'IE0009FT4LX4.EUFUND'},
    'IE00BFG1TM61': {'EODHD': 'IE00BFG1TM61.EUFUND'},
    'IE00BKPTWY98': {'EODHD': 'IE00BKPTWY98.EUFUND'},
    # TUV100 ETFs
    'IE00BFNM3G45': {'EODHD': 'SGAS.XETRA', 'Yahoo': 'SGAS.DE'},
    'IE00BFNM3D14': {'EODHD': 'SLMC.XETRA', 'Yahoo': 'SLMC.DE'},
    'IE00BFNM3L97': {'EODHD': 'SGAJ.XETRA', 'Yahoo': 'SGAJ.DE'},
    # TKF100 ETFs
    'IE00BJZ2DC62': {'EODHD': 'XRSM.XETRA', 'Yahoo': 'XRSM.DE'},
    'IE000O58J820': {'EODHD': 'V3YA.XETRA', 'Yahoo': 'V3YA.DE'},
    'IE000F60HVH9': {'Yahoo': 'USAS.PA'},
    'LU1291106356': {'EODHD': 'PAC.XETRA', 'Yahoo': 'PAC.DE'},
    'IE00BMDBMY19': {'EODHD': 'ESGM.XETRA', 'Yahoo': 'ESGM.DE'},
    'LU1291102447': {'EODHD': 'EJAP.XETRA', 'Yahoo': 'EJAP.DE'},
    'LU1291099718': {'EODHD': 'EEUX.XETRA', 'Yahoo': 'EEUX.DE'},
    'LU0476289540': {'EODHD': 'D5BH.XETRA', 'Yahoo': 'D5BH.DE'},
}

SOURCES = ['EODHD', 'Yahoo']

idx = raw_index.copy()
nav_dt = pd.to_datetime(NAV_DATE)

def exact_price_on_date(key, target_date):
    if key is None:
        return None
    rows = idx[(idx['Key'] == key) & (idx['Date'] == target_date)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]['Value']

def latest_price_on_or_before(key, target_date):
    if key is None:
        return None, None
    rows = idx[(idx['Key'] == key) & (idx['Date'] <= target_date)].sort_values('Date')
    if len(rows) == 0:
        return None, None
    return rows.iloc[-1]['Value'], rows.iloc[-1]['Date']

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']

    verify_rows = []
    stale_warnings = []
    for _, sec in securities.sort_values('Market Value', ascending=False).iterrows():
        isin = sec['Account ID']
        mapping = KEY_MAP.get(isin, {})
        fund_price = sec['Market Price']
        is_stale = isin in STALE_ISINS

        name = sec['Account Name']
        if is_stale:
            name += ' *'

        row = {
            'ISIN': isin, 'Name': name, 'Fund': fund_price,
        }
        for src in SOURCES:
            if is_stale:
                val, val_date = latest_price_on_or_before(mapping.get(src), nav_dt)
                row[src] = val
                if val is not None and val_date is not None and val_date < nav_dt:
                    stale_warnings.append(f'{isin} ({src}): latest price is from {val_date.strftime("%Y-%m-%d")}, not {NAV_DATE}')
            else:
                row[src] = exact_price_on_date(mapping.get(src), nav_dt)
        verify_rows.append(row)

    verify_df = pd.DataFrame(verify_rows)

    def highlight_diff(row):
        fund_val = row['Fund']
        styles = [''] * len(row)
        for i, col in enumerate(row.index):
            if col in SOURCES and pd.notna(row[col]):
                diff_pct = abs(row[col] - fund_val) / fund_val * 100
                if diff_pct > 0.5:
                    styles[i] = 'background-color: #f8d7da'
                elif diff_pct > 0.01:
                    styles[i] = 'background-color: #fff3cd'
                else:
                    styles[i] = 'background-color: #d4edda'
        return styles

    if len(verify_df) > 0:
        display(verify_df.style
            .format({col: '{:.4f}' for col in ['Fund'] + SOURCES}, na_rep='—')
            .set_caption(f'{code} price verification (* = stale, repriced from providers)')
            .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
            .apply(highlight_diff, axis=1)
            .hide(axis='index'))

    # Warn if stale funds don't have fresh prices
    if stale_warnings:
        from IPython.display import display as ipy_display, HTML
        warning_html = '<br>'.join(stale_warnings)
        ipy_display(HTML(
            f'<div style="color: #721c24; background: #f8d7da; padding: 10px 14px; border-radius: 4px; margin: 6px 0 12px 0; font-size: 0.95em">'
            f'<strong>TODO: No fresh prices available for stale pooled funds.</strong><br>'
            f'{warning_html}<br><br>'
            f'EODHD only provides T-1 prices for these IE pooled funds. '
            f'Need to add Morningstar or another source that publishes same-day NAVs for repricing.'
            f'</div>'
        ))

    # Compute repricing adjustment for stale positions
    reprice_adj = 0.0
    for _, vrow in verify_df.iterrows():
        isin = vrow['ISIN']
        fund_price = vrow['Fund']
        alt_prices = [vrow[src] for src in SOURCES if src in vrow and pd.notna(vrow[src])]

        if isin in STALE_ISINS and alt_prices:
            best_price = np.mean(alt_prices)
            qty_row = securities[securities['Account ID'] == isin]
            if len(qty_row) > 0:
                reprice_adj += (best_price - fund_price) * qty_row.iloc[0]['Quantity']

    fd['verify_df'] = verify_df
    fd['reprice_adj'] = reprice_adj
    summary = f'{code}: reprice adj: {reprice_adj:+,.2f} EUR' if abs(reprice_adj) > 0.01 else f'{code}: no repricing needed'
    print(summary)

ISIN,Name,Fund,EODHD,Yahoo
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0 *,15.2390,15.2390,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,33.8700,33.8700,—
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,11.9080,11.9080,—
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible *,13.0600,13.0600,—
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,9.9280,9.9280,—
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,7.3880,7.3880,—


TUV100: no repricing needed


ISIN,Name,Fund,EODHD,Yahoo
IE000F60HVH9,ICAV Amundi MSCI USA Screened UCITS ETF,4.3040,—,—
IE00BJZ2DC62,Xtrackers MSCI USA Screened UCITS ETF,49.7700,49.7700,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,33.8700,33.8700,—
IE000O58J820,Vanguard ESG North America All Cap UCITS ETF,6.7920,6.7920,—
LU1291099718,BNP Paribas Easy MSCI EUROPE MIN TE UCITS ETF,18.6080,18.6080,—
IE00BMDBMY19,Invesco MSCI Emerging Markets Universal Screened UCITS ETF Acc,41.6150,41.6150,—
LU1291102447,BNP Paribas Easy MSCI Japan Min TE UCITS ETF,17.8660,17.8660,—
LU1291106356,BNP Paribas Easy MSCI Pacific ex Japan Min TE UCITS ETF,16.2480,16.2480,—
LU0476289540,Xtrackers MSCI Canada Screened UCITS ETF,103.6000,103.6000,—


TKF100: no repricing needed


## Net Asset Value Calculation

Computing NAV per unit from position data. Fee accrual includes both management fee and depository bank fee (0.025% p.a.), accruing from the start of each month. Stale pooled fund prices are replaced with fresh provider prices.

In [6]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    nav_components = fd['nav_components']
    reprice_adj = fd.get('reprice_adj', 0)
    position_units = fd['position_units']
    mgmt_fee_rate = fd['mgmt_fee_rate']
    units_today = fd['units_today']
    prev_nav_date = fd['prev_nav_date']
    prev_nav_total = fd['prev_nav_total']
    prev_pos = fd['prev_pos']

    # Always use repriced assets
    total_assets = nav_components + reprice_adj

    nav_dt_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')
    day_of_month = nav_dt_dt.day
    total_fee_rate = mgmt_fee_rate + DEPOSITORY_FEE_RATE
    mgmt_accrual = total_assets * mgmt_fee_rate / 365 * day_of_month
    depo_accrual = total_assets * DEPOSITORY_FEE_RATE / 365 * day_of_month
    total_accrual = mgmt_accrual + depo_accrual
    nav_after_fees = total_assets - total_accrual
    computed_nav = nav_after_fees / position_units

    reported_nav = units_today.iloc[0]['Nav'] if len(units_today) > 0 else None
    nav_diff_eur = (computed_nav - reported_nav) if reported_nav else None
    nav_diff_pct = (nav_diff_eur / reported_nav * 100) if reported_nav else None

    # Previous day
    prev_position_units = None
    prev_total_accrual = None
    prev_nav_after_fees = None
    prev_computed_nav = None
    if prev_nav_date and prev_nav_total:
        prev_units_rows = prev_pos[prev_pos['Account Type'] == 'UNITS'] if len(prev_pos) > 0 else pd.DataFrame()
        prev_position_units = prev_units_rows.iloc[0]['Quantity'] if len(prev_units_rows) > 0 else None
        prev_dt = datetime.strptime(prev_nav_date, '%Y-%m-%d')
        prev_total_accrual = prev_nav_total * total_fee_rate / 365 * prev_dt.day
        prev_nav_after_fees = prev_nav_total - prev_total_accrual
        if prev_position_units:
            prev_computed_nav = prev_nav_after_fees / prev_position_units

    def fmt_date(d):
        return datetime.strptime(d, '%Y-%m-%d').strftime('%d.%m.%Y') if d else 'Previous'

    def pct_change(cur, prev):
        if cur is not None and prev is not None and prev != 0:
            return f'{(cur - prev) / abs(prev) * 100:+.2f}%'
        return ''

    col_today = fmt_date(NAV_DATE)
    col_prev = fmt_date(prev_nav_date)

    table_rows = [
        {'': 'Total net assets as reported (EUR)', col_today: f'{nav_components:,.2f}', col_prev: f'{prev_nav_total:,.2f}' if prev_nav_total else '', 'Change': pct_change(nav_components, prev_nav_total)},
    ]
    if abs(reprice_adj) > 0.01:
        table_rows.append(
            {'': 'Repricing adjustment (EUR)', col_today: f'{reprice_adj:+,.2f}', col_prev: '', 'Change': ''},
        )
    table_rows += [
        {'': 'Total net assets repriced (EUR)', col_today: f'{total_assets:,.2f}', col_prev: '', 'Change': ''},
        {'': f'Accrued mgmt fee est. ({mgmt_fee_rate*100:.3f}% p.a.)', col_today: f'{-mgmt_accrual:+,.2f}', col_prev: '', 'Change': ''},
        {'': f'Accrued depository fee est. ({DEPOSITORY_FEE_RATE*100:.3f}% p.a.)', col_today: f'{-depo_accrual:+,.2f}', col_prev: '', 'Change': ''},
        {'': 'Net assets after fees (EUR)', col_today: f'{nav_after_fees:,.2f}', col_prev: f'{prev_nav_after_fees:,.2f}' if prev_nav_after_fees else '', 'Change': pct_change(nav_after_fees, prev_nav_after_fees)},
        {'': 'Units outstanding', col_today: f'{position_units:,.2f}', col_prev: f'{prev_position_units:,.2f}' if prev_position_units else '', 'Change': pct_change(position_units, prev_position_units)},
        {'': 'Computed NAV (EUR)', col_today: f'{computed_nav:.5f}', col_prev: f'{prev_computed_nav:.5f}' if prev_computed_nav else '', 'Change': pct_change(computed_nav, prev_computed_nav)},
    ]
    if reported_nav:
        table_rows += [
            {'': 'Reported NAV (EUR)', col_today: f'{reported_nav:.5f}', col_prev: '', 'Change': ''},
            {'': 'Difference (EUR)', col_today: f'{nav_diff_eur:+.5f}', col_prev: '', 'Change': f'{nav_diff_pct:+.3f}%'},
        ]

    table_df = pd.DataFrame(table_rows)

    ct = col_today
    def style_table(row, col_t=ct):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == col_t:
                styles[i] = 'font-weight: bold'
            elif col_name == 'Change':
                styles[i] = 'font-style: italic'
        return styles

    fee_pct = f'{mgmt_fee_rate * 100:.3f}%' if mgmt_fee_rate else 'N/A'
    caption_html = f'{code} NAV calculation<br><i style="font-weight: normal; font-size: 0.85em">Management fee: {fee_pct} p.a. + Depository fee: {DEPOSITORY_FEE_RATE*100:.3f}% p.a.</i>'

    display(table_df.style
        .set_caption(caption_html)
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(style_table, axis=1)
        .hide(axis='index'))

    fd['reported_nav'] = reported_nav
    fd['computed_nav'] = computed_nav
    fd['prev_computed_nav'] = prev_computed_nav
    fd['nav_diff_eur'] = nav_diff_eur
    fd['nav_diff_pct'] = nav_diff_pct
    fd['total_accrual'] = total_accrual
    fd['total_units'] = position_units

,09.03.2026,06.03.2026,Change
Total net assets as reported (EUR),"475,957,465.11","479,905,546.04",-0.82%
Total net assets repriced (EUR),"475,957,465.11",,
Accrued mgmt fee est. (0.179% p.a.),"-21,007.33",,
Accrued depository fee est. (0.025% p.a.),"-2,933.98",,
Net assets after fees (EUR),"475,933,523.80","479,889,452.77",-0.82%
Units outstanding,"380,461,569.70","380,185,473.76",+0.07%
Computed NAV (EUR),1.25094,1.26225,-0.90%
Reported NAV (EUR),1.25150,,
Difference (EUR),-0.00056,,-0.045%


,09.03.2026,06.03.2026,Change
Total net assets as reported (EUR),"6,757,200.72","6,783,587.61",-0.39%
Total net assets repriced (EUR),"6,757,200.72",,
Accrued mgmt fee est. (0.152% p.a.),-253.26,,
Accrued depository fee est. (0.025% p.a.),-41.65,,
Net assets after fees (EUR),"6,756,905.81","6,783,390.24",-0.39%
Units outstanding,"6,901,876.10","6,888,970.55",+0.19%
Computed NAV (EUR),0.97900,0.98467,-0.58%


## Day-over-Day Comparison

Comparing fund NAV change to benchmark. TUV100: 70% MSCI ACWI Screened + 30% IG3A.DE. TKF: 100% IG3A.DE. All prices are same-day (no lag adjustment needed after repricing).

In [7]:
# TUV100: 70% MSCI ACWI Screened + 30% IG3A.DE (same-day priced after repricing)
TUV100_BENCHMARK = {
    'components': [
        {'key': 'MSCI_ACWI_SCREENED', 'label': 'MSCI ACWI Screened', 'weight': 0.70, 'price_date': 'T'},
        {'key': 'IG3A_YAHOO',         'label': 'IG3A.DE (Yahoo)', 'weight': 0.30, 'price_date': 'T'},
    ],
}

# TKF100: 100% IG3A.DE
TKF100_BENCHMARK = {
    'components': [
        {'key': 'IG3A_YAHOO', 'label': 'IG3A.DE (Yahoo)', 'weight': 1.00, 'price_date': 'T'},
    ],
}

BENCHMARKS = {'TUV100': TUV100_BENCHMARK, 'TKF100': TKF100_BENCHMARK}

idx_data = raw_index.copy()

T = pd.to_datetime(NAV_DATE)
T1 = pd.to_datetime(ee_offset(NAV_DATE, -1))
date_map = {'T': T, 'T1': T1}

def get_daily_change(key, target_date):
    """Get the daily % change for a specific date from the pre-computed series.
    If target_date has no data, use the latest available data point on or before it.
    Returns (change_pct, actual_date, prev_date) or (None, None, None)."""
    series = idx_data[idx_data['Key'] == key].sort_values('Date')
    if len(series) == 0:
        return None, None, None

    exact = series[series['Date'] == target_date]
    if len(exact) > 0 and pd.notna(exact.iloc[0]['DailyChangePct']):
        row = exact.iloc[0]
        idx_pos = series.index.get_loc(exact.index[0])
        prev_dt = series.iloc[idx_pos - 1]['Date'] if idx_pos > 0 else None
        return row['DailyChangePct'], row['Date'], prev_dt

    on_or_before = series[series['Date'] <= target_date]
    if len(on_or_before) < 2:
        return None, None, None
    latest = on_or_before.iloc[-1]
    prev = on_or_before.iloc[-2]
    chg = (latest['Value'] - prev['Value']) / prev['Value'] * 100
    return chg, latest['Date'], prev['Date']

print(f'Pricing dates: T={T.strftime("%Y-%m-%d")}, T-1={T1.strftime("%Y-%m-%d")}\n')

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    prev_date = fd['prev_date']

    print(f'═══ {code} ═══')

    computed_nav = fd.get('computed_nav')
    prev_computed_nav = fd.get('prev_computed_nav')
    reported_nav = fd.get('reported_nav')

    if computed_nav is not None and prev_computed_nav is not None:
        today_nav_val = computed_nav
        prev_nav_val = prev_computed_nav
        nav_source = 'computed from positions'
    elif computed_nav is not None:
        units_prev = fd['units_prev']
        if len(units_prev) > 0:
            today_nav_val = computed_nav
            prev_nav_val = units_prev.iloc[0]['Nav']
            nav_source = 'computed vs prev reported'
        else:
            print(f'Cannot compare: missing previous NAV data\n')
            fd['fund_nav_change_pct'] = None
            continue
    else:
        print(f'Cannot compare: missing data\n')
        fd['fund_nav_change_pct'] = None
        continue

    fund_nav_change_pct = (today_nav_val - prev_nav_val) / prev_nav_val * 100
    n_busdays = ee_busdays(prev_date, NAV_DATE)
    busday_label = f' over {n_busdays} working day{"s" if n_busdays != 1 else ""}'

    print(f'Fund NAV ({nav_source}): {prev_nav_val:.6f} → {today_nav_val:.6f}  ({fund_nav_change_pct:+.2f}%{busday_label})')
    if n_busdays > 1:
        print(f'  Per working day: {fund_nav_change_pct / n_busdays:+.3f}%')

    benchmark = BENCHMARKS[code]
    components = benchmark['components']
    comp_rows = []
    blended_pct = 0
    total_weight = 0
    notices = []

    for comp in components:
        target_date = date_map[comp['price_date']]
        chg, actual_date, prev_dt = get_daily_change(comp['key'], target_date)

        if chg is not None:
            date_label = actual_date.strftime('%m-%d')
            prev_label = prev_dt.strftime('%m-%d') if prev_dt is not None else '?'
            comp_rows.append({
                'Metric': f'{comp["label"]} ({prev_label}→{date_label})',
                'Value': f'{chg:+.2f}%',
                'Weight': f'{comp["weight"]:.0%}',
            })
            blended_pct += comp['weight'] * chg
            total_weight += comp['weight']

            if actual_date != target_date:
                notices.append(f'⚠ {comp["label"]}: no data for {target_date.strftime("%Y-%m-%d")}, using {prev_dt.strftime("%Y-%m-%d")}→{actual_date.strftime("%Y-%m-%d")} instead')
        else:
            comp_rows.append({
                'Metric': f'{comp["label"]} ({target_date.strftime("%Y-%m-%d")})',
                'Value': 'no data',
                'Weight': f'{comp["weight"]:.0%}',
            })

    rows = [
        {'Metric': f'Fund NAV change ({n_busdays}d)', 'Value': f'{fund_nav_change_pct:+.2f}%', 'Weight': ''},
    ]
    if n_busdays > 1:
        rows.append({'Metric': 'Fund NAV change (per day)', 'Value': f'{fund_nav_change_pct / n_busdays:+.3f}%', 'Weight': ''})
    rows.append({'Metric': '', 'Value': '', 'Weight': ''})
    rows.extend(comp_rows)

    if total_weight > 0:
        blended_pct = blended_pct / total_weight
        blended_diff = fund_nav_change_pct - blended_pct
        rows.append({'Metric': 'Blended benchmark', 'Value': f'{blended_pct:+.2f}%', 'Weight': f'{total_weight:.0%}'})
        rows.append({'Metric': 'Tracking diff', 'Value': f'{blended_diff:+.2f}%', 'Weight': ''})
        fd['blended_idx_pct'] = blended_pct
        fd['blended_tracking_diff'] = blended_diff

    comparison = pd.DataFrame(rows)
    display(comparison.style.hide(axis='index'))

    if notices:
        from IPython.display import display as ipy_display, HTML
        ipy_display(HTML('<div style="color: #856404; background: #fff3cd; padding: 8px 12px; border-radius: 4px; margin: 4px 0 12px 0; font-size: 0.9em">' + '<br>'.join(notices) + '</div>'))

    fd['fund_nav_change_pct'] = fund_nav_change_pct
    fd['n_busdays'] = n_busdays
    print()

Pricing dates: T=2026-03-09, T-1=2026-03-06

═══ TUV100 ═══
Fund NAV (computed from positions): 1.262251 → 1.250937  (-0.90% over 1 working day)


Metric,Value,Weight
Fund NAV change (1d),-0.90%,
,,
MSCI ACWI Screened (03-05→03-09),-1.35%,70%
IG3A.DE (Yahoo) (03-06→03-09),-0.45%,30%
Blended benchmark,-1.08%,100%
Tracking diff,+0.18%,



═══ TKF100 ═══
Fund NAV (computed from positions): 0.984674 → 0.978996  (-0.58% over 1 working day)


Metric,Value,Weight
Fund NAV change (1d),-0.58%,
,,
IG3A.DE (Yahoo) (03-06→03-09),-0.45%,100%
Blended benchmark,-0.45%,100%
Tracking diff,-0.13%,
